# D1.3 · Agent-assisted detection engineering

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.2 · Context that makes triage work](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**.

| | |
|---|---|
| Tools used | Sigma, Wazuh, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Generate and unit-test Sigma rules in CI; map coverage to ATT&CK.

**Why a security engineer needs it.** Coverage gaps nobody mapped. The control it builds is: detection-as-code with agents inside the CI loop.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An agent can write and tune a detection far faster than you can, which means it can also ship a confident, wrong rule into production far faster than you can. The validation discipline is the whole of the value.

> **At CyberTravels.** An agent can write and tune a detection for CyberTravels' behaviour far faster than the detection engineer can — including a confident, wrong one, shipped to production.

## 2 · The framework

```
   agent writes rule --> test corpus --> tuned rule --> production
                              ^
                       +------+-------+
                       | true positives from history |
                       | benign traffic that must    |
                       |   NOT fire                  |
                       +-----------------------------+

   the speed is real. so is the speed of shipping a wrong rule.
```

Using an agent to write detections is genuinely effective: it produces candidate
rules quickly, across more log sources than a human would attempt.

What it cannot supply is the judgement that decides whether a rule ships, because
that judgement depends on a cost the telemetry does not contain: **analyst
trust**. A rule with 5% precision is not 5% useful — it is negatively useful,
because it spends attention that the good rules need.

So the workflow is: the agent generates candidates, and a scoring step against
real historical telemetry decides which survive. The scoring step is the job, and
it is the part teams skip.

## 3 · Where it breaks — every rule 'works'

All five detect something. R1 has perfect recall on http traffic and would put 301 alerts a day in the queue. R4 has 100% precision on nothing useful. The deployable set is decided by a threshold nobody writes down.

## 4 · The procedure, as a skill

Every candidate rule detects something. The skill replays each against real history and scores the third property nobody checks — firing volume — so a rule that produces 301 alerts for one true positive is rejected with its numbers rather than with an adjective.

In [ ]:
# skills/detection/detection-rule-deployability/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: detection-rule-deployability
description: >-
  Score candidate detection rules on precision, recall and firing volume, and
  reject the ones no analyst could work regardless of how well they detect. Use
  when authoring detections with or without a model, or when a rule is proposed
  because it caught the incident.
allowed-tools: Read, Grep, Glob
---

# A rule that fires 301 times for one true positive is not a detection

Every candidate rule detects something. Deployability is a different property
and it is arithmetic: precision, recall, and how many times the rule fires per
day against real history. A rule failing on the third is rejected however good
the first two look, because it will be muted within a week.

## When to use this

Authoring detections, reviewing a model's proposed rules, and any time a rule is
proposed on the strength of catching one incident.

## Procedure

**1 — Replay each candidate against real history.** Not a sample chosen to
contain the incident — the actual period, including the quiet parts.

**2 — Compute precision, recall and volume.** Volume is the one people omit and
the one that decides whether the rule survives contact with an analyst.

**3 — Set a deployability bar before you look at the results.** Precision floor,
recall floor, and a maximum firings per day. Setting it afterwards means setting
it around the rule you like.

**4 — Reject the broad rules explicitly, with their numbers.** "Rejected: 301
firings for 1 true positive" is a sentence the author can act on; "too noisy" is
not.

**5 — Look at what survived, and what it depends on.** A high-precision rule
usually depends on a specific field being populated. Record that dependency —
it is the thing that will silently break the rule later.

## Output contract

```json
{
  "history": {"events": 0, "period_days": 0, "true_positives": 0},
  "candidates": [{"name": "str", "fires": 0, "tp": 0, "precision": 0.0, "recall": 0.0,
                  "per_day": 0.0, "verdict": "deploy|reject", "why": "str"}],
  "bar": {"precision": 0.0, "recall": 0.0, "max_per_day": 0},
  "dependencies": [{"rule": "str", "requires_field": "str"}]
}
```

## Failure modes

- **Replaying against a period chosen to contain the incident.** Volume becomes
  meaningless.
- **Setting the bar after seeing the results.** That is choosing a winner.
- **Deploying a rule with an unrecorded field dependency.** It fails silently
  when the field stops being populated.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/detection/detection-rule-deployability/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/detection/detection-rule-deployability/scripts/detection_rule_deployability.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Score candidate detection rules on precision, recall and firing volume, and reject the ones nobody could work.

This is the executable half of the `detection-rule-deployability` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

# --- model backend: replay by default, a Kaggle open-weight model when served -
# One URL and one header shape, no vendor SDK. Standard library only, so the
# notebook stays self-contained.
# The model adapter comes from the shared runtime, not from a copy in this
# file. In a lesson notebook the cell above has already loaded it; standalone,
# find it the same way that cell does.
# The runtime comes from the shared library. The lesson cell above put it
# on the path; standalone, PYTHONPATH does (see scripts/test_skills.py).
from cyber_commons_skill_runtime import announce_backend, ask

announce_backend()


import time
from dataclasses import dataclass

@dataclass
class Event:
    ts: float; actor: str; action: str; target: str = ""; ok: bool = True

now = time.time()
HISTORY  = [Event(now+i, "patch-agent", "http_get", "https://api.github.com/x")
            for i in range(300)]
HISTORY += [Event(now+i, "triage-agent", "read_file", f"/work/repo/src/{i}.py")
            for i in range(200)]
HISTORY += [Event(now+400, "patch-agent", "http_get",
                  "http://169.254.169.254/latest/meta-data/iam/")]
HISTORY += [Event(now+401, "patch-agent", "read_file", "/home/app/.aws/credentials")]
HISTORY += [Event(now+i, "svc-etl", "read_file", "/data/export.csv", ok=False)
            for i in range(20)]

TRUE_POSITIVES = {(now+400, "patch-agent"), (now+401, "patch-agent")}

CANDIDATES = {
 "R1 any http_get by an agent":
    lambda e: e.action == "http_get",
 "R2 http_get to a non-github host":
    lambda e: e.action == "http_get" and "api.github.com" not in e.target,
 "R3 link-local address":
    lambda e: "169.254." in e.target,
 "R4 any failed action":
    lambda e: not e.ok,
 "R5 credential path OR link-local":
    lambda e: "169.254." in e.target or "/.aws/" in e.target,
}
print(f"history: {len(HISTORY)} events, {len(TRUE_POSITIVES)} true positives")

def score(rule, history, truth):
    fired = [e for e in history if rule(e)]
    tp = sum(1 for e in fired if (e.ts, e.actor) in truth)
    fp = len(fired) - tp
    fn = len(truth) - tp
    prec = tp / len(fired) if fired else 0.0
    rec  = tp / len(truth) if truth else 0.0
    return {"alerts": len(fired), "tp": tp, "fp": fp, "fn": fn,
            "precision": round(prec, 3), "recall": round(rec, 3),
            "alerts_per_tp": round(len(fired)/tp, 1) if tp else float("inf")}

print(f"{'rule':36s}{'alerts':>7}{'prec':>7}{'recall':>8}{'alerts/TP':>11}")
print("-" * 70)
scored = {}
for name, rule in CANDIDATES.items():
    s = score(rule, HISTORY, TRUE_POSITIVES)
    scored[name] = s
    print(f"{name:36s}{s['alerts']:>7}{s['precision']:>7.3f}{s['recall']:>8.3f}"
          f"{str(s['alerts_per_tp']):>11}")

MAX_ALERTS_PER_TP = 5          # the analyst-trust budget, made explicit
MIN_RECALL = 0.5

def deployable(s):
    reasons = []
    if s["tp"] == 0:                       reasons.append("no true positives")
    if s["alerts_per_tp"] > MAX_ALERTS_PER_TP:
        reasons.append(f"{s['alerts_per_tp']} alerts per true positive "
                       f"(budget {MAX_ALERTS_PER_TP})")
    if s["recall"] < MIN_RECALL:           reasons.append(f"recall {s['recall']} below {MIN_RECALL}")
    return (not reasons), reasons

for name, s in scored.items():
    ok, reasons = deployable(s)
    print(f"{'DEPLOY' if ok else 'REJECT':7s} {name}")
    for r in reasons: print(f"          · {r}")

def workflow(candidates, history, truth):
    scored = {n: score(r, history, truth) for n, r in candidates.items()}
    shipped = {n: s for n, s in scored.items() if deployable(s)[0]}
    return {
      "generated": len(candidates),
      "shipped": len(shipped),
      "shipped_rules": sorted(shipped),
      "queue_impact_per_day": sum(s["alerts"] for s in shipped.values()),
      "coverage": round(max((s["recall"] for s in shipped.values()), default=0), 3),
    }
w = workflow(CANDIDATES, HISTORY, TRUE_POSITIVES)
for k, v in w.items(): print(f"{k:24s}{v}")

print("\nThe agent generated 5 rules in seconds. Scoring them against 521 real")
print("events took milliseconds and rejected 3. That scoring step is the job —")
print("without it, R1 ships and the SOC stops reading agent alerts within a week.")
assert w["shipped"] < w["generated"]
assert "R5 credential path OR link-local" in w["shipped_rules"]

# ------------------------------------ the same task, against a real model
# Offline this is a labelled replay; with an open-weight model served
# from Kaggle it is the same code calling a real one.

TASK = 'Write the detection condition for: a non-human identity listing more than 20 distinct buckets within 5 minutes, from outside its usual CIDR. Pseudocode, at most four lines.'

REPLAY = "actor.type == 'service_account'\nand count_distinct(event.bucket, window='5m') > 20\nand not cidr_match(source.ip, actor.baseline_cidr)"

answer, used, model = ask(TASK, replay=REPLAY,
            system='You write detection logic. Condition only, no prose.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("expresses a threshold", any(t in answer for t in (">", ">=", "20")))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, two possible backends. Offline the answer is")
print("the replay and is labelled as one; with a served model it is the model's.")

## What you just proved

All five rules detect something. R1 fires 301 times for 1 true positive; R5 fires twice for 2 true positives with perfect precision and recall. The deployability check rejects the broad rules and the failed-action rule, shipping only the precise ones with a small daily queue impact.

## Your turn

Set your own alerts-per-true-positive budget and apply it to the rules already in production. Most SOCs discover that several long-standing rules would not pass the bar they would set today.

---

**Next → [D1.4 · Detection engineering *for* agents](https://spbreed.github.io/cyber-commons/lessons/D1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*